In [1]:
import event_helpers
import gcsfs
import geopandas as gpd
import google.auth
import pandas as pd
import poi_geography_utils as poi_geog_utils
import world_cup_vars as wc_vars
from gtfs_curator_utils import utils

GCS_FILE_PATH = wc_vars.GCS_FILE_PATH
credentials, _ = google.auth.default()

In [2]:
STADIUM_NAME = "sofi"
stop_gdf = gpd.read_parquet(
    f"{GCS_FILE_PATH}stop_summary_{STADIUM_NAME}.parquet",
    storage_options = {"token": credentials.token}
)

In [3]:
import create_route_and_stop_event_summary as A1

routes_df = A1.import_routes_near_poi(wc_vars.event_name, "sofi")

stop_gdf = A1.prep_fct_daily_scheduled_stops(wc_vars.event_name, wc_vars.sofi_match_times).merge(
    routes_df[["feed_key", "schedule_name"]].drop_duplicates(),
    how="inner",
)

In [4]:
# get non-event 
arrivals_cols = [c for c in stop_gdf.columns if "arrivals_per_hour" in c]

nonevent_arrivals_df = (
    event_helpers.aggregate_by_event_type(
        stop_gdf[~stop_gdf.service_date.isin(wc_vars.sofi_dates)],
        group_cols=[
            "schedule_name",
            "stop_id",
            "stop_name",
            "event_day",
            "day_type",
        ],
        mean_cols=arrivals_cols
    )
)

/tmp/ipykernel_330/2911718186.py:6: FutureWarning: The behavior of 'isin' with dtype=datetime64[ns] and castable values (e.g. strings) is deprecated. In a future version, these will not be considered matching by isin. Explicitly cast to the appropriate dtype before calling isin instead.
  stop_gdf[~stop_gdf.service_date.isin(wc_vars.sofi_dates)],


In [5]:
arrivals_by_event = pd.concat(
    [nonevent_arrivals_df,
     stop_gdf[stop_gdf.service_date.isin(wc_vars.sofi_dates)]
    ], axis=0, ignore_index=True
)

/tmp/ipykernel_330/3619135629.py:3: FutureWarning: The behavior of 'isin' with dtype=datetime64[ns] and castable values (e.g. strings) is deprecated. In a future version, these will not be considered matching by isin. Explicitly cast to the appropriate dtype before calling isin instead.
  stop_gdf[stop_gdf.service_date.isin(wc_vars.sofi_dates)]


In [6]:
arrivals_by_event_long = event_helpers.make_long(
    arrivals_by_event, 
    id_vars = ["service_date", "schedule_name", "stop_id", "stop_name", "day_type"] # service_date for non_event are NaT
)

In [7]:
# Aggregate this across number of stops for operator
# Keep average arrivals_per_hour
arrivals_by_event_long2 = (
    arrivals_by_event_long
    .groupby(["service_date", "schedule_name", "time_of_day", "day_type"], dropna=False)
    .agg({
        "arrivals_per_hour": "mean",
        "stop_id": "count"
    }).reset_index().rename(columns = {"stop_id": "n_stops"})
)

arrivals_by_event_long2 = arrivals_by_event_long2.assign(
    service_date = arrivals_by_event_long2.service_date.dt.date.fillna('nonevent').astype(str),
)

In [76]:
import altair as alt

alt.data_transformers.enable("vegafusion")

def bar_chart_by_time_of_day(df: pd.DataFrame, color_col: str):
    
    selection = alt.selection_point(fields=[color_col], bind="legend")
    
    chart = (
        alt.Chart(df)
        .mark_bar()
        .encode(
            x="arrivals_per_hour",
            y=alt.Y("time_of_day", sort=event_helpers.TIME_OF_DAY_ORDER),
            tooltip = ["schedule_name", "day_type", "time_of_day", "arrivals_per_hour", "n_stops"],
            color=alt.Color(f"{color_col}:N", sort=event_helpers.TIME_OF_DAY_ORDER),
            opacity=alt.when(selection).then(alt.value(1)).otherwise(alt.value(0.2)),        
        )
    )
    
    return chart

### Sofi weekday midday matches

In [77]:
time_of_day = "midday"
day_type = "weekday"

def concat_bar_chart(df: pd.DataFrame, color_col: str, 
                     time_of_day: str, day_type: str):

    filtered_sofi_dict = event_helpers.grab_matches_by_day_type(
        wc_vars.sofi_match_times, time_of_day, day_type)
    
    indiv_dates = list(filtered_sofi_dict.keys())

    selection = alt.selection_point(fields=[color_col], bind="legend")

    nonevent_chart = bar_chart_by_time_of_day(
        df[(df.service_date == "nonevent") & (df.day_type==day_type)], color_col=color_col
    ).properties(title = {
        "text": f"Typical {day_type} arrivals",
    }).add_params(selection)
    
    indiv_date_charts = [
        bar_chart_by_time_of_day(
            df[df.service_date == d], color_col=color_col
        ).properties(title = {
            "text": f"SoFi Match: {d} at {time_of_day}",
            "subtitle": f"{day_type}"
        }).add_params(selection) for d in indiv_dates
    ]    

    return [nonevent_chart] + indiv_date_charts 
    

In [78]:
event_charts = concat_bar_chart(arrivals_by_event_long2, "time_of_day", "midday", "weekday")

In [79]:
alt.vconcat(*event_charts).resolve_scale(x="shared")

/home/jovyan/gtfs-curator/.venv/lib/python3.11/site-packages/IPython/core/interactiveshell.py:3699: UserWarning: Automatically deduplicated selection parameter with identical configuration. If you want independent parameters, explicitly name them differently (e.g., name='param1', name='param2'). See https://github.com/vega/altair/issues/3891
  exec(code_obj, self.user_global_ns, self.user_ns)
/home/jovyan/gtfs-curator/.venv/lib/python3.11/site-packages/IPython/core/interactiveshell.py:3699: UserWarning: Automatically deduplicated selection parameter with identical configuration. If you want independent parameters, explicitly name them differently (e.g., name='param1', name='param2'). See https://github.com/vega/altair/issues/3891
  exec(code_obj, self.user_global_ns, self.user_ns)
/home/jovyan/gtfs-curator/.venv/lib/python3.11/site-packages/IPython/core/interactiveshell.py:3699: UserWarning: Automatically deduplicated selection parameter with identical configuration. If you want indepe

alt.VConcatChart(...)

In [89]:
filtered_sofi_dict = event_helpers.grab_matches_by_day_type(
    wc_vars.sofi_match_times, time_of_day, day_type)

indiv_dates = list(filtered_sofi_dict.keys())
print(indiv_dates)

alt.Chart(
    arrivals_by_event_long2[
        (arrivals_by_event_long2.time_of_day=="midday") & 
        (arrivals_by_event_long2.day_type=="weekday") & 
        (arrivals_by_event_long2.service_date.isin(["nonevent"] + indiv_dates)
        )
        ]
).mark_bar().encode(
    x="arrivals_per_hour",
    y="service_date",
    color="schedule_name"
)

['2026-06-18', '2026-07-02', '2026-07-10']


alt.Chart(...)

In [ ]:
bar_chart_by_time_of_day(
    arrivals_by_event_long2[arrivals_by_event_long2.service_date == "nonevent"]
)

In [ ]:
alt.vconcat(
    non_event_chart, event_jun18_chart, 
    event_jul2_chart, event_jul10_chart
).resolve_scale(x="shared")